In [ ]:
import os
import mne
import warnings
import scipy.signal as signal
import numpy as np
from typing import List, Dict, Tuple
import import_ipynb
import eeg_preprocessing_interface as eeg_pp
import eeg_band_separation as eeg_band
import eeg_complexity_feture as eeg_feture
from tqdm import tqdm
from scipy.io import savemat 

# Critical configuration: Disable truncation and print all elements
np.set_printoptions(
    threshold=np.inf,  # Remove element count limit (print all)
    linewidth=1000     # Show more elements per line, reduce line breaks (optional)
)
mne.set_log_level("ERROR")
warnings.filterwarnings(
    "ignore",  # Action: ignore warnings
    category=RuntimeWarning,  # Warning type: RuntimeWarning
    message="This filename.*does not conform to MNE naming conventions"  # Warning keyword (regex match)
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="NOTE: pick_channels\\(\\) is a legacy function. New code should use inst.pick\\(\\.\\.\\.\\)."
)

In [ ]:
# 1. Basic Configuration (Subjects, Directories, Time Points, etc.)
sub_10hz = ["100306", "100412", "100515", "100723", "100927", "101029", "101139","101449", 
            "101551", "101656", "101758", "101861", "101965", "102274", "102375", "102478", "102580"]

sub_sham = ["300207", "300308", "300409", "300618", "300720","300822", "300925", "301041", "301250",
            "301354", "301455", "301560", "301662", "301766", "301867", "302177", "302282", "302384"]  # Removed "300513"

# Directory Configuration
base_dir_10hz = r'D:\山东第一医科大学\数据\Task\八因子\10hz'
base_dir_sham = r'D:\山东第一医科大学\数据\Task\八因子\sham'
out_base_dir = r'D:\山东第一医科大学\数据\Task\八因子\脑网络'  # Can be used for saving results later if needed
target_bands = ['delta', 'theta', 'alpha', 'lbeta', 'hbeta', 'gamma1', 'gamma2', 'gamma3', 'gamma4', 'gamma5', 'gamma6']

# Experiment Dimension Configuration
time_points = ['post','pre'] 
emotions = ['sad']
groups = {
    #'10hz': {'subjects': sub_10hz, 'source_dir': base_dir_10hz},
    'sham': {'subjects': sub_sham, 'source_dir': base_dir_sham}
}
extractor = eeg_feture.EEGComplexityAnalyzer()

In [ ]:
def compute_plv_matrix_optimized(filtered_data):
    """
    Optimized PLV matrix calculation (reduced memory usage)
    filtered_data: shape (n_channels, n_samples)
    return: PLV matrix (n_channels, n_channels)
    """
    n_channels = filtered_data.shape[0]
    plv_matrix = np.zeros((n_channels, n_channels), dtype=np.float32)  # Use float32 to reduce memory
    
    # Compute PLV for each channel pair individually (avoid large (n_ch, n_ch, n_samples) matrix)
    for i in range(n_channels):
        # Compute analytic signal and phase for channel i
        analytic_i = signal.hilbert(filtered_data[i], axis=-1)
        phase_i = np.angle(analytic_i)
        
        for j in range(i+1, n_channels):  # Upper triangular matrix, reduce redundant calculations
            # Compute phase for channel j
            analytic_j = signal.hilbert(filtered_data[j], axis=-1)
            phase_j = np.angle(analytic_j)
            
            # Complex average of phase differences
            phase_diff = phase_i - phase_j
            plv = np.abs(np.mean(np.exp(1j * phase_diff), axis=-1))
            
            plv_matrix[i, j] = plv
            plv_matrix[j, i] = plv  # Symmetric matrix
    
    np.fill_diagonal(plv_matrix, 0)  # Set diagonal to 0 (no self-connection)
    return plv_matrix

In [ ]:
# Add progress bar for group loop
for group_name, group_info in tqdm(groups.items(), desc="Processing Groups", unit="group"):
    source_dir = group_info['source_dir']
    subject_list = group_info['subjects']  # 20 subjects
    
    # Add progress bar for time point loop
    for time in tqdm(time_points, desc=f"Processing {group_name} Time Points", unit="time point", leave=False):
        # Add progress bar for emotion loop
        for emotion in tqdm(emotions, desc=f"Processing {time} Emotions", unit="emotion", leave=False):
            
            # 3. Iterate over subjects with progress bar
            sub_file = []
            for sub_idx, sub in enumerate(tqdm(subject_list, desc="Processing Subjects", unit="subject", leave=False)):
                sub_source_dir = os.path.join(source_dir, time, emotion, sub)
                print(f"Current directory: {sub_source_dir}")
                 # Iterate over frequency bands
                band_file = [] 
                for band_idx, band_name in enumerate(target_bands):
                    # Iterate over factors
                    factor_file = []
                    for factor_num in range(1, 9):
                        factor_filename = f"factor_{factor_num}.fif"
                        factor_path = os.path.join(sub_source_dir, band_name, factor_filename)

                        # Check if file exists
                        if not os.path.exists(factor_path):
                            print(f"Warning: File not found, skipping -> {factor_path}")
                            continue  # Skip missing files and continue
                        
                        raw = mne.io.read_raw_fif(factor_path, preload=True, verbose=False)
                        band_np_data = raw.get_data()  # Shape: (n_channels, n_samples)
                        plv_matrix = compute_plv_matrix_optimized(band_np_data)

                        factor_file.append(plv_matrix)    
                    band_file.append(np.array(factor_file).mean(axis = 0))
                sub_file.append(band_file)
                
            filename = f'{group_name}_{time}_{emotion}.npy'
            outpath = os.path.join(out_base_dir,filename)
            print(np.array(sub_file).shape)
            np.save(outpath, np.array(sub_file))